The live ShopFlow event stream: RATE orders per second for SECONDS seconds.

In [ ]:
import random, sys, time

In [2]:
from confluent_kafka import Producer

In [3]:
%run order.ipynb

In [ ]:
first = 1
rate = 1
seconds = 1

In [ ]:
producer = Producer({
    "bootstrap.servers": "localhost:9092",
    "acks": "all",
    "enable.idempotence": True,
    "linger.ms": 20,
})

In [ ]:
rng = random.Random(first)
sent, bad = 0, 0

In [ ]:
for n in range(first , first + rate * seconds):
    order = make_order(n, rng)
    if n % 50 == 0:
        order["qty"] = 0
        bad += 1
    
    producer.produce("orders", key=order["customer_id"], value=encode(order))
    sent += 1
    if sent % rate == 0:
        producer.flush()
        print(f"{sent} orders send, latest {order["ts"]} {order["city"]}", flush=True)
        time.sleep(1)

producer.flush()
print(f"stream finished: {sent} orders, {bad} of them broken on purpose")

In [ ]:
# Watch the stream in a separate terminal
# > docker exec --workdir /opt/kafka/bin/ -it broker sh
# kafka> kafka-console-consumer.sh --bootstrap-server $BS --topic orders --formatter-property print.key=true

In [ ]:
# Start the stream
# > papermill .\shop_stream.ipynb .\output_shop_stream.ipynb -p first 1001 -p rate 10 -p seconds 3